<a href="https://colab.research.google.com/github/ghadirymaryam1-coder/PFMS-Net-Research2026/blob/main/PFMSPARALLEL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch, os, shutil

print("Python environment ready")
print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")
print("Local free space (GB):", round(shutil.disk_usage("/content").free / 1e9, 2))
print("Drive mounted:", os.path.exists("/content/drive/MyDrive"))

Python environment ready
PyTorch: 2.11.0+cu130
CUDA: 13.0
GPU: Tesla T4
Local free space (GB): 75.31
Drive mounted: False


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
import os

paths = {
    "GT CSV": "/content/drive/MyDrive/IDPL_PFOD_REAL_GT_MAPPING.csv",
    "Vocabulary": "/content/drive/MyDrive/idpl_vocab.pt",
    "Images": "/content/drive/Othercomputers/My Laptop/Desktop/idplimgl",
}

for name, path in paths.items():
    print(f"{name}: {'FOUND' if os.path.exists(path) else 'MISSING'}")

GT CSV: FOUND
Vocabulary: FOUND
Images: FOUND


In [7]:
%pip install --no-cache-dir --no-deps --force-reinstall "causal-conv1d==1.6.2.post1+cu.13.0.torch.2.11" --index-url https://wheels.astral.sh/simple/cu130/

Looking in indexes: https://wheels.astral.sh/simple/cu130/
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 205.6/205.6 MB 155.5 MB/s  0:00:01


In [8]:
%pip install --no-cache-dir --no-deps --force-reinstall "mamba-ssm==2.3.2.post1+cu.13.0.torch.2.11" --index-url https://wheels.astral.sh/simple/cu130/

Looking in indexes: https://wheels.astral.sh/simple/cu130/
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.4/349.4 MB 186.3 MB/s  0:00:01


In [10]:
%pip install --no-cache-dir --no-deps mambavision==1.2.0

In [11]:
from mambavision import create_model
import torch
import causal_conv1d
import mamba_ssm

model = create_model("mamba_vision_T", pretrained=False).cuda().eval()

with torch.no_grad():
    y = model(torch.randn(1, 3, 224, 224, device="cuda"))

print("MambaVision: OK")
print("GPU:", torch.cuda.get_device_name(0))
print("Output shape:", y.shape)
print("Finite output:", torch.isfinite(y).all().item())

del model, y
torch.cuda.empty_cache()

/usr/local/lib/python3.13/dist-packages/timm/models/registry.py:4: FutureWarning: Importing from timm.models.registry is deprecated, please import via timm.models
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.models", FutureWarning)
/usr/local/lib/python3.13/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


MambaVision: OK
GPU: Tesla T4
Output shape: torch.Size([1, 1000])
Finite output: True


In [12]:
import os

paths = {
    "GT": "/content/drive/MyDrive/IDPL_PFOD_REAL_GT_MAPPING.csv",
    "Vocabulary": "/content/drive/MyDrive/idpl_vocab.pt",
    "Images": "/content/drive/Othercomputers/My Laptop/Desktop/idplimgl",
}

for name, path in paths.items():
    print(f"{name}: {'OK' if os.path.exists(path) else 'NOT FOUND'}")

GT: OK
Vocabulary: OK
Images: OK


In [16]:
# ============================================================
# PFMS-Net Ablation | Cell 2: Spatial feature-map verification
# Real MambaVision-T + original IDPL-PFOD images
# ============================================================
import os
import torch
import pandas as pd
import numpy as np
from PIL import Image
from collections import Counter
from mambavision import create_model

GT_PATH = "/content/drive/MyDrive/IDPL_PFOD_REAL_GT_MAPPING.csv"
VOCAB_PATH = "/content/drive/MyDrive/idpl_vocab.pt"
OUT_DIR = "/content/PFMS_ABLATION"
os.makedirs(OUT_DIR, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
gt = pd.read_csv(GT_PATH)
train_df = gt[gt["split"].astype(str).str.lower() == "train"].reset_index(drop=True)
test_df = gt[gt["split"].astype(str).str.lower() == "test"].reset_index(drop=True)

# Inspect the saved vocabulary without assuming its internal format.
vocab_obj = torch.load(VOCAB_PATH, map_location="cpu", weights_only=False)
print("Saved vocabulary object type:", type(vocab_obj).__name__)

if isinstance(vocab_obj, dict):
    print("Vocabulary keys (first 30):", list(vocab_obj.keys())[:30])
    for k, v in list(vocab_obj.items())[:10]:
        if isinstance(v, dict):
            print(f"  {k}: dict, entries={len(v)}")
        elif isinstance(v, (list, tuple)):
            print(f"  {k}: {type(v).__name__}, length={len(v)}")
        else:
            print(f"  {k}: {type(v).__name__}", str(v)[:120])
elif isinstance(vocab_obj, (list, tuple)):
    print("Vocabulary length:", len(vocab_obj))
else:
    print("Vocabulary preview:", str(vocab_obj)[:500])

print(f"\nTrain rows: {len(train_df)} | Test rows: {len(test_df)}")
print("Unique training characters:", len(set("".join(train_df["text"].astype(str)))))

# Load a real image and inspect dimensions.
sample_path = str(train_df.iloc[0]["image_path"])
if not os.path.isfile(sample_path):
    raise FileNotFoundError(f"Image path from GT does not exist: {sample_path}")

with Image.open(sample_path) as im:
    im = im.convert("RGB")
    print("\nSample image:", os.path.basename(sample_path))
    print("Original image size (W,H):", im.size)

# Build the actual MambaVision-T backbone.
mamba = create_model("mamba_vision_T", pretrained=False).to(device).eval()
captured = {}

def capture_stage(name):
    def _hook(module, inputs, output):
        captured[name] = output
    return _hook

handles = []
for i, stage in enumerate(mamba.levels):
    handles.append(stage.register_forward_hook(capture_stage(f"stage_{i+1}")))

# Probe with the original aspect ratio scaled down only for this shape test.
# This is not training and does not resize any dataset files on disk.
probe_h, probe_w = 64, 448
x = torch.randn(1, 3, probe_h, probe_w, device=device)

with torch.no_grad():
    _ = mamba(x)

for h in handles:
    h.remove()

def describe_output(obj):
    if torch.is_tensor(obj):
        return f"Tensor{tuple(obj.shape)}"
    if isinstance(obj, (tuple, list)):
        return type(obj).__name__ + "[" + ", ".join(describe_output(z) for z in obj) + "]"
    return type(obj).__name__

print("\nMambaVision internal stage outputs:")
for name, obj in captured.items():
    print(name, "->", describe_output(obj))

# Choose a spatial tensor from the deepest stage output.
last_output = captured[f"stage_{len(mamba.levels)}"]
candidates = []
def collect_spatial(obj):
    if torch.is_tensor(obj) and obj.ndim == 4:
        candidates.append(obj)
    elif isinstance(obj, (tuple, list)):
        for z in obj:
            collect_spatial(z)
collect_spatial(last_output)

if not candidates:
    raise RuntimeError(
        "No 4D spatial feature map was exposed by the last stage. "
        "Do not train using the pooled 640-D classifier features."
    )

feature_map = max(candidates, key=lambda z: z.shape[1])
print("\nSelected spatial feature map:", tuple(feature_map.shape))
print("Spatial width (sequence axis):", feature_map.shape[-1])
print("Spatial height:", feature_map.shape[-2])
print("Feature channels:", feature_map.shape[1])

# Save only a small text report locally.
report_path = os.path.join(OUT_DIR, "mambavision_feature_probe.txt")
with open(report_path, "w", encoding="utf-8") as f:
    f.write(f"Train rows: {len(train_df)}\n")
    f.write(f"Test rows: {len(test_df)}\n")
    f.write(f"Probe input: {tuple(x.shape)}\n")
    f.write(f"Selected feature map: {tuple(feature_map.shape)}\n")
    f.write(f"Vocabulary type: {type(vocab_obj).__name__}\n")

print("\nProbe report saved locally:", report_path)
print("No checkpoint or feature cache was written to Google Drive.")

Saved vocabulary object type: dict
Vocabulary keys (first 30): ['char2idx', 'idx2char', 'characters', 'PAD_IDX', 'BOS_IDX', 'EOS_IDX', 'UNK_IDX', 'vocab_size']
  char2idx: dict, entries=169
  idx2char: dict, entries=169
  characters: list, length=165
  PAD_IDX: int 0
  BOS_IDX: int 1
  EOS_IDX: int 2
  UNK_IDX: int 3
  vocab_size: int 169

Train rows: 23054 | Test rows: 4066
Unique training characters: 164

Sample image: 00490.tif
Original image size (W,H): (700, 50)

MambaVision internal stage outputs:
stage_1 -> Tensor(1, 160, 8, 56)
stage_2 -> Tensor(1, 320, 4, 28)
stage_3 -> Tensor(1, 640, 2, 14)
stage_4 -> Tensor(1, 640, 2, 14)

Selected spatial feature map: (1, 640, 2, 14)
Spatial width (sequence axis): 14
Spatial height: 2
Feature channels: 640

Probe report saved locally: /content/PFMS_ABLATION/mambavision_feature_probe.txt
No checkpoint or feature cache was written to Google Drive.


In [17]:
# ============================================================
# PFMS-Net Ablation | Cell 3: OCR data pipeline + CTC sanity test
# ============================================================
import os, gc, random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from torchvision import transforms
from mambavision import create_model

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda")
GT_PATH = "/content/drive/MyDrive/IDPL_PFOD_REAL_GT_MAPPING.csv"
VOCAB_PATH = "/content/drive/MyDrive/idpl_vocab.pt"
OUT_DIR = "/content/PFMS_ABLATION"
os.makedirs(OUT_DIR, exist_ok=True)

gt = pd.read_csv(GT_PATH)
train_df = gt[gt["split"].astype(str).str.lower() == "train"].reset_index(drop=True)
test_df = gt[gt["split"].astype(str).str.lower() == "test"].reset_index(drop=True)

vocab = torch.load(VOCAB_PATH, map_location="cpu", weights_only=False)
char2idx = vocab["char2idx"]
PAD_IDX = int(vocab["PAD_IDX"])
BOS_IDX = int(vocab["BOS_IDX"])
EOS_IDX = int(vocab["EOS_IDX"])
UNK_IDX = int(vocab["UNK_IDX"])
VOCAB_SIZE = int(vocab["vocab_size"])

# CTC blank must be distinct from actual target characters.
# PAD=0 is used as CTC blank; BOS/EOS are excluded from targets.
CTC_BLANK = PAD_IDX

# Resize the line to a fixed landscape size, keeping the line layout.
# Image files on Drive are never modified.
IMG_H, IMG_W = 64, 896
transform = transforms.Compose([
    transforms.Resize((IMG_H, IMG_W)),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.485, 0.456, 0.406),
                         std=(0.229, 0.224, 0.225)),
])

class IDPLLineDataset(Dataset):
    def __init__(self, frame):
        self.frame = frame

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, idx):
        row = self.frame.iloc[idx]
        path = str(row["image_path"])
        text = str(row["text"]).replace("\n", " ").strip()
        with Image.open(path) as im:
            image = transform(im.convert("RGB"))

        # CTC targets: character IDs only, no BOS/EOS.
        target = [char2idx.get(ch, UNK_IDX) for ch in text]
        target = [t for t in target if t not in (PAD_IDX, BOS_IDX, EOS_IDX)]
        return image, torch.tensor(target, dtype=torch.long), text, path

def ctc_collate(batch):
    images, targets, texts, paths = zip(*batch)
    target_lengths = torch.tensor([len(t) for t in targets], dtype=torch.long)
    flat_targets = torch.cat(targets) if target_lengths.sum().item() else torch.empty(0, dtype=torch.long)
    return torch.stack(images), flat_targets, target_lengths, list(texts), list(paths)

# Keep test data separate; no test examples used for optimization.
train_ds = IDPLLineDataset(train_df)
test_ds = IDPLLineDataset(test_df)

train_loader = DataLoader(
    train_ds, batch_size=4, shuffle=True, num_workers=2,
    collate_fn=ctc_collate, pin_memory=True
)
test_loader = DataLoader(
    test_ds, batch_size=4, shuffle=False, num_workers=2,
    collate_fn=ctc_collate, pin_memory=True
)

# Real MambaVision backbone
backbone = create_model("mamba_vision_T", pretrained=False).to(device).eval()
stage4 = backbone.levels[-1]
captured = {}

def save_stage_output(module, inputs, output):
    captured["feat"] = output

hook = stage4.register_forward_hook(save_stage_output)

# Use one real batch to verify CTC dimensions before training.
images, targets, target_lengths, texts, paths = next(iter(train_loader))
images = images.to(device, non_blocking=True)

with torch.no_grad():
    _ = backbone(images)
hook.remove()

feat = captured["feat"]
if not torch.is_tensor(feat) or feat.ndim != 4:
    raise RuntimeError(f"Expected spatial feature map [B,C,H,W], got {type(feat)}")

# Convert spatial map [B,C,H,W] -> sequence [T,B,C].
# Pool only vertical dimension, preserving the horizontal sequence.
feat = feat.mean(dim=2).permute(2, 0, 1).contiguous()
sequence_length = feat.shape[0]
batch_size = feat.shape[1]

print("Vocabulary size:", VOCAB_SIZE)
print("Image batch:", tuple(images.shape))
print("MambaVision sequence [T,B,C]:", tuple(feat.shape))
print("CTC blank index:", CTC_BLANK)
print("Target lengths:", target_lengths.tolist())
print("Text samples:", texts[:2])

if sequence_length < int(target_lengths.max().item()):
    print(
        "\nWARNING: CTC sequence is shorter than at least one target. "
        "Do not start training with this output stride. We must increase "
        "the horizontal sequence resolution before training."
    )
else:
    logits = torch.randn(sequence_length, batch_size, VOCAB_SIZE, device=device)
    log_probs = logits.log_softmax(dim=-1)
    input_lengths = torch.full((batch_size,), sequence_length, dtype=torch.long)
    loss_fn = nn.CTCLoss(blank=CTC_BLANK, zero_infinity=True)
    loss = loss_fn(log_probs, targets.to(device), input_lengths, target_lengths)
    print("CTC sanity loss finite:", bool(torch.isfinite(loss)))
    print("CTC sanity loss:", float(loss))

del backbone, images, feat, captured
gc.collect()
torch.cuda.empty_cache()
print("\nData pipeline verified. No training was started in this cell.")

Vocabulary size: 169
Image batch: (4, 3, 64, 896)
MambaVision sequence [T,B,C]: (28, 4, 640)
CTC blank index: 0
Target lengths: [67, 77, 72, 72]
Text samples: ['دشمنان در كمین هستند و تا ظهور امام عصر این انقلاب را به پیش ببریم.', 'به شهروندان مهم و قابل تحسین دانست. به گزارش تحریرنو از روابط عمومی بانک شهر،']


Data pipeline verified. No training was started in this cell.


In [18]:
# ============================================================
# MambaVision OCR: find a suitable spatial stage for CTC
# No training and no Drive writes
# ============================================================
import torch
import gc
from mambavision import create_model

device = "cuda"
model = create_model("mamba_vision_T", pretrained=False).to(device).eval()

captured = {}
handles = []

def make_hook(name):
    def hook(module, inputs, output):
        captured[name] = output
    return hook

for i, stage in enumerate(model.levels):
    handles.append(stage.register_forward_hook(make_hook(f"stage_{i+1}")))

# 64x896 line image, matching the tested OCR input shape
x = torch.randn(1, 3, 64, 896, device=device)

with torch.no_grad():
    _ = model(x)

for h in handles:
    h.remove()

print("Stage | Feature map [B,C,H,W] | CTC steps after vertical pooling")
for name, value in captured.items():
    if torch.is_tensor(value) and value.ndim == 4:
        print(f"{name}: {tuple(value.shape)} | T={value.shape[-1]}")

# Stage 2 is the first candidate: higher horizontal resolution
feat = captured["stage_2"]
assert feat.ndim == 4, f"Unexpected stage_2 output: {type(feat)}"
print("\nCandidate stage_2 channels:", feat.shape[1])
print("Candidate CTC steps:", feat.shape[-1])
print("Longest sample target length in previous batch: 77")
print("CTC minimum feasibility requires input steps >= target length.")
print("For repeated adjacent characters, additional steps may be required.")

del model, x, feat, captured
gc.collect()
torch.cuda.empty_cache()

Stage | Feature map [B,C,H,W] | CTC steps after vertical pooling
stage_1: (1, 160, 8, 112) | T=112
stage_2: (1, 320, 4, 56) | T=56
stage_3: (1, 640, 2, 28) | T=28
stage_4: (1, 640, 2, 28) | T=28

Candidate stage_2 channels: 320
Candidate CTC steps: 56
Longest sample target length in previous batch: 77
CTC minimum feasibility requires input steps >= target length.
For repeated adjacent characters, additional steps may be required.


In [19]:
# ============================================================
# PFMS-Net | MambaVision spatial-resolution check
# ============================================================
import torch
import pandas as pd
from mambavision import create_model

GT_PATH = "/content/drive/MyDrive/IDPL_PFOD_REAL_GT_MAPPING.csv"
df = pd.read_csv(GT_PATH)
train_df = df[df["split"].astype(str).str.lower() == "train"].copy()

vocab = torch.load(
    "/content/drive/MyDrive/idpl_vocab.pt",
    map_location="cpu",
    weights_only=False
)
char2idx = vocab["char2idx"]
excluded = {
    int(vocab["PAD_IDX"]),
    int(vocab["BOS_IDX"]),
    int(vocab["EOS_IDX"])
}
unk = int(vocab["UNK_IDX"])

def target_len(text):
    return sum(
        1 for ch in str(text).replace("\n", " ").strip()
        if char2idx.get(ch, unk) not in excluded
    )

lengths = train_df["text"].map(target_len)
print("Train samples:", len(train_df))
print("Target length max:", int(lengths.max()))
print("Target length p99:", int(lengths.quantile(0.99)))
print("Targets longer than 128:", int((lengths > 128).sum()))

model = create_model("mamba_vision_T", pretrained=False).cuda().eval()
captured = {}

def hook_stage1(module, inputs, output):
    captured["stage1"] = output

h = model.levels[0].register_forward_hook(hook_stage1)
x = torch.randn(1, 3, 64, 1024, device="cuda")

with torch.no_grad():
    _ = model(x)

h.remove()
feat = captured["stage1"]
print("Stage-1 spatial feature map:", tuple(feat.shape))
print("CTC sequence length:", int(feat.shape[-1]))
print("CTC blank:", vocab["PAD_IDX"])

assert feat.ndim == 4
assert feat.shape[-1] >= int(lengths.max()), (
    "CTC sequence is shorter than the longest target. "
    "Increase width or use a different spatial feature strategy."
)

print("Resolution check passed for target lengths.")
del model, x, feat, captured
torch.cuda.empty_cache()

Train samples: 23054
Target length max: 114
Target length p99: 97
Targets longer than 128: 0
Stage-1 spatial feature map: (1, 160, 8, 128)
CTC sequence length: 128
CTC blank: 0
Resolution check passed for target lengths.


In [21]:

# CELL 1/2 — PFMS-Net REAL ablation: data, backbones, CTC checks
import os, sys, json, time, random, math, csv, copy, gc, warnings, subprocess
from pathlib import Path

# Install only if a required import is missing; do not reinstall PyTorch.
def ensure_import(module, pip_name):
    try:
        __import__(module)
    except Exception:
        subprocess.check_call([sys.executable, "-m", "pip", "install", pip_name])

ensure_import("timm", "timm")
ensure_import("mambavision", "mambavision")
import numpy as np
import pandas as pd
from PIL import Image, ImageOps
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import timm
from mambavision import create_model

# -------------------- Fixed experiment settings --------------------
SEED = 42
BATCH_SIZE = 4
EPOCHS = 10
LR = 1e-4
WEIGHT_DECAY = 1e-2
IMG_H, IMG_W = 64, 1024
SEQ_LEN = 128
D_MODEL = 256
NUM_WORKERS = 2
USE_AMP = True

CSV_PATH = "/content/drive/MyDrive/IDPL_PFOD_REAL_GT_MAPPING.csv"
VOCAB_PATH = "/content/drive/MyDrive/idpl_vocab.pt"
IMAGE_ROOT = "/content/drive/Othercomputers/My Laptop/Desktop/idplimgl"
OUT_DIR = Path("/content/PFMS_ABLATION_FINAL")
OUT_DIR.mkdir(parents=True, exist_ok=True)

assert torch.cuda.is_available(), "GPU لازم است: Runtime > Change runtime type > GPU"
DEVICE = torch.device("cuda")
torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True

def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything()
print("Torch:", torch.__version__, "| GPU:", torch.cuda.get_device_name(0))
print("Output directory:", OUT_DIR)

# -------------------- Load and validate vocabulary --------------------
vocab_obj = torch.load(VOCAB_PATH, map_location="cpu", weights_only=False)
if isinstance(vocab_obj, dict) and "char2idx" in vocab_obj:
    char2idx = vocab_obj["char2idx"]
    idx2char = vocab_obj.get("idx2char", None)
    PAD_IDX = int(vocab_obj.get("PAD_IDX", 0))
    BOS_IDX = int(vocab_obj.get("BOS_IDX", 1))
    EOS_IDX = int(vocab_obj.get("EOS_IDX", 2))
    UNK_IDX = int(vocab_obj.get("UNK_IDX", 3))
else:
    raise ValueError(
        "ساختار idpl_vocab.pt با کلید char2idx تطبیق ندارد؛ "
        "ساختار واقعی فایل را بررسی کنید."
    )

char2idx = {str(k): int(v) for k, v in char2idx.items()}
if idx2char is None:
    idx2char = {v: k for k, v in char2idx.items()}
else:
    idx2char = {int(k): str(v) for k, v in idx2char.items()}

VOCAB_SIZE = max(max(char2idx.values()), max(idx2char.keys())) + 1
assert PAD_IDX == 0 and BOS_IDX == 1 and EOS_IDX == 2 and UNK_IDX == 3
assert VOCAB_SIZE == 169, f"Expected vocab size 169; got {VOCAB_SIZE}"

# CTC targets must NOT contain padding, BOS, or EOS.
SPECIAL_IDS = {PAD_IDX, BOS_IDX, EOS_IDX}
BLANK_IDX = PAD_IDX

def encode_text(text):
    ids = [char2idx.get(ch, UNK_IDX) for ch in str(text)]
    ids = [i for i in ids if i not in SPECIAL_IDS]
    if not ids:
        raise ValueError(f"Empty target after removing special tokens: {text!r}")
    if any(i < 0 or i >= VOCAB_SIZE for i in ids):
        raise ValueError("Target token ID outside vocabulary.")
    return ids

def decode_ctc(ids):
    """Greedy CTC decoding: merge repeats first, then remove blank."""
    out, previous = [], None
    for token in ids:
        token = int(token)
        if token != previous and token != BLANK_IDX:
            out.append(idx2char.get(token, "�"))
        previous = token
    return "".join(out)

# -------------------- Read CSV and validate split --------------------
df = pd.read_csv(CSV_PATH)
required_cols = {"split", "image_path", "image_name", "text"}
assert required_cols.issubset(df.columns), (
    f"Missing CSV columns: {required_cols - set(df.columns)}"
)
df["split"] = df["split"].astype(str).str.lower().str.strip()
df["text"] = df["text"].fillna("").astype(str)

def resolve_image_path(row):
    p = Path(str(row["image_path"]))
    if p.is_file():
        return str(p)
    p2 = Path(IMAGE_ROOT) / str(row["image_name"])
    if p2.is_file():
        return str(p2)
    p3 = Path(IMAGE_ROOT) / p.name
    if p3.is_file():
        return str(p3)
    return str(p)

df["resolved_path"] = df.apply(resolve_image_path, axis=1)
df = df[df["text"].str.len() > 0].copy()
missing = df.loc[~df["resolved_path"].map(lambda p: Path(p).is_file())]
if len(missing):
    raise FileNotFoundError(
        f"{len(missing)} images not found. Example paths:\n"
        + "\n".join(missing["resolved_path"].head(5).tolist())
    )

# Use only official train split for train/validation.
train_all = df[df["split"].str.contains("train")].copy()
test_df = df[df["split"].str.contains("test")].copy()
assert len(train_all) > 0 and len(test_df) > 0
assert not set(train_all.index).intersection(set(test_df.index))

# Fixed, reproducible 90/10 split of the official training data.
perm = np.random.RandomState(SEED).permutation(len(train_all))
n_val = max(1, int(round(0.10 * len(train_all))))
val_idx, tr_idx = perm[:n_val], perm[n_val:]
train_df = train_all.iloc[tr_idx].reset_index(drop=True)
val_df = train_all.iloc[val_idx].reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

# Preflight every target, including CTC repeated-character minimum length.
def ctc_min_steps(ids):
    return len(ids) + sum(ids[i] == ids[i-1] for i in range(1, len(ids)))

for name, part in [("train", train_df), ("val", val_df), ("test", test_df)]:
    enc = [encode_text(t) for t in part["text"]]
    max_len = max(map(len, enc))
    max_min_steps = max(map(ctc_min_steps, enc))
    print(f"{name}: n={len(part)}, max_target={max_len}, "
          f"max_CTC_min_steps={max_min_steps}")
    if max_min_steps > SEQ_LEN:
        raise ValueError(
            f"{name} contains a target requiring {max_min_steps} CTC steps, "
            f"but SEQ_LEN={SEQ_LEN}. Increase input width/sequence length."
        )

print("Vocabulary:", VOCAB_SIZE, "| CTC blank:", BLANK_IDX)
print("Train/validation/test:", len(train_df), len(val_df), len(test_df))

# -------------------- Dataset: printed Persian text lines --------------------
IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
IMAGENET_STD = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

class PersianLineDataset(Dataset):
    def __init__(self, frame):
        self.frame = frame.reset_index(drop=True)

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, index):
        row = self.frame.iloc[index]
        with Image.open(row["resolved_path"]) as im:
            im = ImageOps.exif_transpose(im).convert("RGB")
            # Fixed geometry for all ablation variants.
            im = im.resize((IMG_W, IMG_H), Image.Resampling.BILINEAR)
            arr = np.asarray(im, dtype=np.float32) / 255.0
        x = torch.from_numpy(arr).permute(2, 0, 1)
        x = (x - IMAGENET_MEAN) / IMAGENET_STD
        target = torch.tensor(encode_text(row["text"]), dtype=torch.long)
        return x, target, str(row["text"]), str(row["image_name"])

def collate_lines(batch):
    images, targets, texts, names = zip(*batch)
    lengths = torch.tensor([len(t) for t in targets], dtype=torch.long)
    return (
        torch.stack(images),
        torch.cat(targets),
        lengths,
        list(texts),
        list(names),
    )

def make_loader(frame, shuffle):
    return DataLoader(
        PersianLineDataset(frame),
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
        num_workers=NUM_WORKERS,
        pin_memory=True,
        collate_fn=collate_lines,
        persistent_workers=(NUM_WORKERS > 0),
    )

train_loader = make_loader(train_df, True)
val_loader = make_loader(val_df, False)
test_loader = make_loader(test_df, False)

# -------------------- Real visual encoders --------------------
class SwinStage1(nn.Module):
    """Real timm Swin feature map, no ImageNet classification head."""
    def __init__(self):
        super().__init__()
        self.net = timm.create_model(
            "swin_tiny_patch4_window7_224",
            pretrained=False,
            features_only=True,
            out_indices=(0,),
            img_size=(IMG_H, IMG_W),
        )
        self.channels = int(self.net.feature_info.channels()[0])

    def forward(self, x):
        feat = self.net(x)[0]  # [B,C,H',W']
        if feat.ndim != 4:
            raise RuntimeError(f"Swin must return NCHW map, got {feat.shape}")
        return feat

class RealMambaVisionStage1(nn.Module):
    """
    Real MambaVision stem + first actual stage.
    This is deliberately a stage-1 spatial encoder, not the full four-stage
    MambaVision model. The 1000-class head and later stages are not used.
    """
    def __init__(self):
        super().__init__()
        backbone = create_model("mamba_vision_T", pretrained=False)
        if not hasattr(backbone, "patch_embed") or not hasattr(backbone, "levels"):
            raise RuntimeError("Unexpected MambaVision API: patch_embed/levels missing")
        self.patch_embed = backbone.patch_embed
        self.stage1 = backbone.levels[0]
        del backbone
        self.channels = 160

    def forward(self, x):
        x = self.patch_embed(x)
        x = self.stage1(x)
        if x.ndim != 4:
            raise RuntimeError(f"MambaVision stage-1 must return NCHW, got {x.shape}")
        if x.shape[1] != self.channels:
            raise RuntimeError(
                f"Expected MambaVision stage-1 channels={self.channels}, got {x.shape}"
            )
        return x

# -------------------- Fusion + CTC OCR heads --------------------
class CrossAttentionBlock(nn.Module):
    def __init__(self, dim=D_MODEL, heads=4, dropout=0.1):
        super().__init__()
        self.norm_q = nn.LayerNorm(dim)
        self.norm_kv = nn.LayerNorm(dim)
        self.attn = nn.MultiheadAttention(
            dim, heads, dropout=dropout, batch_first=True
        )
        self.ffn = nn.Sequential(
            nn.LayerNorm(dim),
            nn.Linear(dim, dim * 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(dim * 2, dim),
        )
        self.drop = nn.Dropout(dropout)

    def forward(self, q, kv):
        qn = self.norm_q(q)
        kvn = self.norm_kv(kv)
        z, _ = self.attn(qn, kvn, kvn, need_weights=False)
        q = q + self.drop(z)
        return q + self.drop(self.ffn(q))

class OCRAblationModel(nn.Module):
    def __init__(self, variant):
        super().__init__()
        valid = {"swin_only", "mamba_only", "simple_fusion",
                 "cross_attention", "full_pfms"}
        assert variant in valid
        self.variant = variant
        self.swin = SwinStage1() if variant != "mamba_only" else None
        self.mamba = RealMambaVisionStage1() if variant != "swin_only" else None

        if self.swin is not None:
            self.swin_proj = nn.Conv2d(self.swin.channels, D_MODEL, 1)
        if self.mamba is not None:
            self.mamba_proj = nn.Conv2d(self.mamba.channels, D_MODEL, 1)

        if variant == "simple_fusion":
            self.fuse = nn.Sequential(
                nn.Linear(D_MODEL * 2, D_MODEL), nn.GELU(),
                nn.LayerNorm(D_MODEL),
            )
        if variant in {"cross_attention", "full_pfms"}:
            self.swin_to_mamba = CrossAttentionBlock()
            self.mamba_to_swin = CrossAttentionBlock()

        self.refine = None
        if variant == "full_pfms":
            self.refine = nn.Sequential(
                nn.Conv1d(D_MODEL, D_MODEL, 3, padding=1),
                nn.GELU(),
                nn.Conv1d(D_MODEL, D_MODEL, 3, padding=1),
            )

        self.classifier = nn.Linear(D_MODEL, VOCAB_SIZE)

    @staticmethod
    def spatial_to_sequence(feat, proj):
        feat = proj(feat)                     # [B,D,H,W]
        feat = feat.mean(dim=2)               # [B,D,W]
        feat = F.adaptive_avg_pool1d(feat, SEQ_LEN)
        return feat.transpose(1, 2).contiguous()  # [B,T,D]

    def forward(self, images):
        s = None if self.swin is None else self.spatial_to_sequence(
            self.swin(images), self.swin_proj
        )
        m = None if self.mamba is None else self.spatial_to_sequence(
            self.mamba(images), self.mamba_proj
        )

        if self.variant == "swin_only":
            z = s
        elif self.variant == "mamba_only":
            z = m
        elif self.variant == "simple_fusion":
            z = self.fuse(torch.cat([s, m], dim=-1))
        else:
            # Bidirectional cross-attention: each branch queries the other.
            s2 = self.swin_to_mamba(s, m)
            m2 = self.mamba_to_swin(m, s)
            z = 0.5 * (s2 + m2)
            if self.refine is not None:
                z = z + self.refine(z.transpose(1, 2)).transpose(1, 2)

        logits = self.classifier(z)             # [B,T,V]
        if logits.shape[1] != SEQ_LEN or logits.shape[2] != VOCAB_SIZE:
            raise RuntimeError(f"Bad CTC logits shape: {logits.shape}")
        return logits

# -------------------- Architecture/CTC preflight BEFORE training --------------------
def run_preflight():
    x = torch.zeros(1, 3, IMG_H, IMG_W, device=DEVICE)
    variants = ["swin_only", "mamba_only", "simple_fusion",
                "cross_attention", "full_pfms"]
    results = []
    for variant in variants:
        model = OCRAblationModel(variant).to(DEVICE).eval()
        with torch.no_grad():
            y = model(x)
        assert y.shape == (1, SEQ_LEN, VOCAB_SIZE), (variant, y.shape)
        params = sum(p.numel() for p in model.parameters())
        trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
        results.append({
            "variant": variant, "logits_shape": str(tuple(y.shape)),
            "parameters_including_unused": params,
            "trainable_parameters": trainable,
        })
        print("PREFLIGHT OK:", variant, tuple(y.shape),
              "| trainable params:", f"{trainable:,}")
        del model, y
        gc.collect()
        torch.cuda.empty_cache()
    pd.DataFrame(results).to_csv(OUT_DIR / "preflight.csv", index=False)
    del x
    return results

PREFLIGHT_RESULTS = run_preflight()
print("Preflight passed. No training has been run in Cell 1.")


Torch: 2.11.0+cu130 | GPU: Tesla T4
Output directory: /content/PFMS_ABLATION_FINAL
train: n=20749, max_target=115, max_CTC_min_steps=116
val: n=2305, max_target=109, max_CTC_min_steps=111
test: n=4066, max_target=105, max_CTC_min_steps=107
Vocabulary: 169 | CTC blank: 0
Train/validation/test: 20749 2305 4066


RuntimeError: Given groups=1, weight of size [256, 96, 1, 1], expected input[1, 16, 256, 96] to have 96 channels, but got 16 channels instead

In [ ]:

# CELL 2/2 — Train, validate, test, and export real ablation results
import gc, time, json, math
from pathlib import Path

# Explicitly check that Cell 1 completed successfully.
assert len(PREFLIGHT_RESULTS) == 5
assert all(r["logits_shape"] == str((1, SEQ_LEN, VOCAB_SIZE))
           for r in PREFLIGHT_RESULTS)

VARIANTS = [
    "swin_only",
    "mamba_only",
    "simple_fusion",
    "cross_attention",
    "full_pfms",
]

# CTC loss: blank is PAD_IDX=0, but PAD/BOS/EOS are absent from targets.
ctc_loss_fn = nn.CTCLoss(
    blank=BLANK_IDX,
    reduction="mean",
    zero_infinity=False,
)

def edit_distance(a, b):
    """Memory-efficient Levenshtein distance."""
    if len(a) < len(b):
        a, b = b, a
    previous = list(range(len(b) + 1))
    for i, ca in enumerate(a, start=1):
        current = [i]
        for j, cb in enumerate(b, start=1):
            current.append(min(
                current[-1] + 1,
                previous[j] + 1,
                previous[j - 1] + (ca != cb),
            ))
        previous = current
    return previous[-1]

def text_metrics(references, predictions):
    char_edits = char_count = 0
    word_edits = word_count = 0
    exact = 0

    for ref, pred in zip(references, predictions):
        char_edits += edit_distance(list(ref), list(pred))
        char_count += max(1, len(ref))
        rw, pw = ref.split(), pred.split()
        word_edits += edit_distance(rw, pw)
        word_count += max(1, len(rw))
        exact += int(ref == pred)

    cer = char_edits / char_count
    wer = word_edits / word_count
    n = max(1, len(references))
    return {
        "CER": cer,
        "WER": wer,
        "character_accuracy": 1.0 - cer,
        "word_accuracy": 1.0 - wer,
        "sequence_accuracy": exact / n,
        "char_edits": char_edits,
        "reference_characters": char_count,
        "word_edits": word_edits,
        "reference_words": word_count,
        "exact_sequences": exact,
        "samples": len(references),
    }

def autocast_context():
    return torch.autocast(
        device_type="cuda",
        dtype=torch.float16,
        enabled=USE_AMP,
    )

def count_parameters(model):
    return sum(p.numel() for p in model.parameters())

def evaluate(model, loader, collect_predictions=False):
    model.eval()
    total_loss = 0.0
    total_samples = 0
    refs, preds, pred_rows = [], [], []
    elapsed = 0.0

    with torch.inference_mode():
        for images, targets, target_lengths, texts, names in loader:
            images = images.to(DEVICE, non_blocking=True)
            targets = targets.to(DEVICE, non_blocking=True)
            target_lengths = target_lengths.to(DEVICE, non_blocking=True)
            bsz = images.shape[0]

            if DEVICE.type == "cuda":
                torch.cuda.synchronize()
            start = time.perf_counter()

            with autocast_context():
                logits = model(images)

            # Keep CTC log-probabilities in float32 for numerical stability.
            log_probs = logits.float().log_softmax(dim=-1)
            input_lengths = torch.full(
                (bsz,), log_probs.shape[1],
                dtype=torch.long, device="cpu"
            )
            loss = ctc_loss_fn(
                log_probs.transpose(0, 1),
                targets,
                input_lengths,
                target_lengths.cpu(),
            )

            if DEVICE.type == "cuda":
                torch.cuda.synchronize()
            elapsed += time.perf_counter() - start

            total_loss += float(loss.item()) * bsz
            total_samples += bsz

            decoded_ids = log_probs.argmax(dim=-1).cpu().tolist()
            batch_preds = [decode_ctc(ids) for ids in decoded_ids]
            refs.extend(texts)
            preds.extend(batch_preds)

            if collect_predictions:
                for name, ref, pred in zip(names, texts, batch_preds):
                    pred_rows.append({
                        "image_name": name,
                        "reference": ref,
                        "prediction": pred,
                        "exact_match": ref == pred,
                    })

    metrics = text_metrics(refs, preds)
    metrics["loss"] = total_loss / max(1, total_samples)
    metrics["inference_ms_per_image"] = (
        1000.0 * elapsed / max(1, total_samples)
    )
    metrics["images_per_second"] = total_samples / max(elapsed, 1e-12)

    if collect_predictions:
        return metrics, pd.DataFrame(pred_rows)
    return metrics

def train_one_variant(variant):
    print("\n" + "=" * 80)
    print("TRAINING:", variant)
    print("=" * 80)

    # Identical initialization seed policy for each variant.
    seed_everything(SEED)
    model = OCRAblationModel(variant).to(DEVICE)
    params_total = count_parameters(model)
    params_trainable = sum(
        p.numel() for p in model.parameters() if p.requires_grad
    )

    optimizer = torch.optim.AdamW(
        model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY
    )
    # Separate scaler per model.
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)

    model_dir = OUT_DIR / variant
    model_dir.mkdir(parents=True, exist_ok=True)
    best_path = model_dir / "best_model.pt"
    last_path = model_dir / "last_model.pt"
    history = []
    best_val_loss = float("inf")

    for epoch in range(1, EPOCHS + 1):
        model.train()
        running_loss = 0.0
        seen = 0
        epoch_start = time.perf_counter()

        for step, (images, targets, target_lengths, texts, names) in enumerate(
            train_loader, start=1
        ):
            images = images.to(DEVICE, non_blocking=True)
            targets = targets.to(DEVICE, non_blocking=True)
            target_lengths = target_lengths.to(DEVICE, non_blocking=True)
            bsz = images.shape[0]

            optimizer.zero_grad(set_to_none=True)

            with autocast_context():
                logits = model(images)

            log_probs = logits.float().log_softmax(dim=-1)
            input_lengths = torch.full(
                (bsz,), log_probs.shape[1],
                dtype=torch.long, device="cpu"
            )

            # Runtime safety: no target may require more CTC steps than available.
            required_steps = [
                ctc_min_steps(encode_text(t)) for t in texts
            ]
            if max(required_steps) > log_probs.shape[1]:
                raise RuntimeError(
                    f"Invalid CTC alignment in {variant}: "
                    f"required={max(required_steps)}, available={log_probs.shape[1]}"
                )

            loss = ctc_loss_fn(
                log_probs.transpose(0, 1),
                targets,
                input_lengths,
                target_lengths.cpu(),
            )

            if not torch.isfinite(loss):
                raise FloatingPointError(
                    f"Non-finite CTC loss: model={variant}, "
                    f"epoch={epoch}, batch={step}"
                )

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            scaler.step(optimizer)
            scaler.update()

            running_loss += float(loss.item()) * bsz
            seen += bsz

            if step % 100 == 0 or step == len(train_loader):
                print(
                    f"{variant} | epoch {epoch}/{EPOCHS} "
                    f"| batch {step}/{len(train_loader)} "
                    f"| loss={loss.item():.5f}"
                )

        train_loss = running_loss / max(1, seen)
        val_metrics = evaluate(model, val_loader)
        epoch_seconds = time.perf_counter() - epoch_start

        record = {
            "variant": variant,
            "epoch": epoch,
            "train_loss": train_loss,
            "val_loss": val_metrics["loss"],
            "val_CER": val_metrics["CER"],
            "val_WER": val_metrics["WER"],
            "val_character_accuracy": val_metrics["character_accuracy"],
            "val_word_accuracy": val_metrics["word_accuracy"],
            "val_sequence_accuracy": val_metrics["sequence_accuracy"],
            "epoch_seconds": epoch_seconds,
            "parameters_total": params_total,
            "parameters_trainable": params_trainable,
            "seed": SEED,
            "batch_size": BATCH_SIZE,
            "learning_rate": LR,
        }
        history.append(record)
        pd.DataFrame(history).to_csv(
            model_dir / "training_history.csv", index=False
        )

        # Save the latest state and the best checkpoint based on validation loss.
        checkpoint = {
            "variant": variant,
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "val_loss": val_metrics["loss"],
            "seed": SEED,
            "vocab_size": VOCAB_SIZE,
            "blank_idx": BLANK_IDX,
            "sequence_length": SEQ_LEN,
            "image_height": IMG_H,
            "image_width": IMG_W,
            "model_description": (
                "Swin stage-1 / MambaVision stem+stage-1; "
                "CTC OCR; see experiment configuration."
            ),
        }
        torch.save(checkpoint, last_path)

        if val_metrics["loss"] < best_val_loss:
            best_val_loss = val_metrics["loss"]
            torch.save(checkpoint, best_path)

        print(
            f"Epoch {epoch}: train_loss={train_loss:.5f}, "
            f"val_loss={val_metrics['loss']:.5f}, "
            f"val_CER={val_metrics['CER']:.5f}, "
            f"val_WER={val_metrics['WER']:.5f}, "
            f"time={epoch_seconds:.1f}s"
        )

    # Reload the best validation checkpoint before evaluating the held-out test.
    del optimizer, scaler
    gc.collect()
    torch.cuda.empty_cache()

    checkpoint = torch.load(best_path, map_location=DEVICE, weights_only=False)
    model.load_state_dict(checkpoint["model_state_dict"])
    test_metrics, pred_df = evaluate(
        model, test_loader, collect_predictions=True
    )

    pred_df.to_csv(model_dir / "test_predictions.csv", index=False)
    with open(model_dir / "test_metrics.json", "w", encoding="utf-8") as f:
        json.dump(test_metrics, f, ensure_ascii=False, indent=2)

    summary = {
        "variant": variant,
        "best_epoch": int(checkpoint["epoch"]),
        "best_val_loss": float(checkpoint["val_loss"]),
        "parameters_total": params_total,
        "parameters_trainable": params_trainable,
        "test_loss": test_metrics["loss"],
        "CER": test_metrics["CER"],
        "WER": test_metrics["WER"],
        "character_accuracy": test_metrics["character_accuracy"],
        "word_accuracy": test_metrics["word_accuracy"],
        "sequence_accuracy": test_metrics["sequence_accuracy"],
        "inference_ms_per_image": test_metrics["inference_ms_per_image"],
        "images_per_second": test_metrics["images_per_second"],
        "test_samples": test_metrics["samples"],
        "seed": SEED,
        "epochs_requested": EPOCHS,
        "batch_size": BATCH_SIZE,
        "learning_rate": LR,
        "weight_decay": WEIGHT_DECAY,
        "image_height": IMG_H,
        "image_width": IMG_W,
        "ctc_sequence_length": SEQ_LEN,
        "vocab_size": VOCAB_SIZE,
        "checkpoint": str(best_path),
        "prediction_file": str(model_dir / "test_predictions.csv"),
    }

    with open(model_dir / "summary.json", "w", encoding="utf-8") as f:
        json.dump(summary, f, ensure_ascii=False, indent=2)

    print("\nHELD-OUT TEST:", json.dumps(summary, ensure_ascii=False, indent=2))
    del model, checkpoint
    gc.collect()
    torch.cuda.empty_cache()
    return summary

# Save configuration first, so the exact intended protocol is recorded.
config = {
    "seed": SEED, "batch_size": BATCH_SIZE, "epochs": EPOCHS,
    "optimizer": "AdamW", "learning_rate": LR,
    "weight_decay": WEIGHT_DECAY, "loss": "CTCLoss",
    "blank_idx": BLANK_IDX, "vocab_size": VOCAB_SIZE,
    "image_size": [IMG_H, IMG_W], "ctc_sequence_length": SEQ_LEN,
    "variants": VARIANTS,
    "mamba_encoder_note": (
        "Real MambaVision stem + first stage only; not the complete "
        "four-stage backbone."
    ),
    "swin_encoder": "timm swin_tiny_patch4_window7_224, stage-1 features",
    "validation_policy": "10% deterministic holdout from official train split",
    "test_policy": "held out from optimization and model selection",
}
with open(OUT_DIR / "experiment_config.json", "w", encoding="utf-8") as f:
    json.dump(config, f, ensure_ascii=False, indent=2)

all_summaries = []
for variant in VARIANTS:
    summary = train_one_variant(variant)
    all_summaries.append(summary)
    pd.DataFrame(all_summaries).to_csv(
        OUT_DIR / "ablation_results_partial.csv", index=False
    )

results = pd.DataFrame(all_summaries)
results.to_csv(OUT_DIR / "ablation_results.csv", index=False)

# Paper-friendly percentage table; raw results remain in [0,1] in CSV.
paper_table = results[[
    "variant", "parameters_total", "CER", "WER",
    "character_accuracy", "word_accuracy", "sequence_accuracy",
    "inference_ms_per_image", "images_per_second",
]].copy()

for col in [
    "CER", "WER", "character_accuracy",
    "word_accuracy", "sequence_accuracy",
]:
    paper_table[col] = 100.0 * paper_table[col]

paper_table = paper_table.rename(columns={
    "variant": "Model",
    "parameters_total": "Parameters",
    "CER": "CER (%)",
    "WER": "WER (%)",
    "character_accuracy": "Character accuracy (%)",
    "word_accuracy": "Word accuracy (%)",
    "sequence_accuracy": "Sequence accuracy (%)",
    "inference_ms_per_image": "Inference (ms/image)",
    "images_per_second": "Throughput (images/s)",
})
paper_table.to_csv(OUT_DIR / "ablation_paper_table.csv", index=False)

print("\nFINAL ABLATION TABLE — measured results only")
display(paper_table.round(4))
print("\nAll outputs saved to:", OUT_DIR)
print("Do not mix these results with metrics from earlier experiments.")
